In [19]:
import pymupdf

pdf_path = "Introducing-MLOps.pdf"

doc = pymupdf.open(pdf_path)

raw_text = ""
for page in doc:
    if page.number < 15 or page.number > 171:
        continue

    blocks = page.get_text("blocks")  # returns (x0, y0, x1, y1, text, ...)
    for block in blocks:
        y0, y1 = block[1], block[3]
        page_height = page.rect.height
        # skip top 8% and bottom 8% of page
        # if y0 > page_height * 0.08 and y1 < page_height * 0.92:
        if y1 < page_height * 0.92:
            raw_text += block[4]


print(f"Total number of characters: {len(raw_text)}")
print(raw_text[:100], end="\n\n")
print(raw_text[-100:])

doc.close()

Total number of characters: 327060
CHAPTER 1
Why Now and Challenges
Machine learning operations (MLOps) is quickly becoming a critical 

llows data teams (and more widely, entire organiza‐
tions) to scale their machine learning efforts.



In [2]:
import re

# words = re.split(r'(\s)', raw_text)
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [word for word in preprocessed if word.strip()]

print(f"Total length of preprocessed words: {len(preprocessed)}")
print(preprocessed[:99])

Total length of preprocessed words: 58589
['CHAPTER', '1', 'Why', 'Now', 'and', 'Challenges', 'Machine', 'learning', 'operations', '(', 'MLOps', ')', 'is', 'quickly', 'becoming', 'a', 'critical', 'component', 'of', 'successful', 'data', 'science', 'project', 'deployment', 'in', 'the', 'enterprise', '(', 'Figure', '1-1', ')', '.', 'It’s', 'a', 'process', 'that', 'helps', 'organizations', 'and', 'business', 'leaders', 'generate', 'long-term', 'value', 'and', 'reduce', 'risk', 'associated', 'with', 'data', 'science', ',', 'machine', 'learning', ',', 'and', 'AI', 'initiatives', '.', 'Yet', 'it’s', 'a', 'rela‐', 'tively', 'new', 'concept', ';', 'so', 'why', 'has', 'it', 'seemingly', 'skyrocketed', 'into', 'the', 'data', 'science', 'lexicon', 'overnight', '?', 'This', 'introductory', 'chapter', 'delves', 'into', 'what', 'MLOps', 'is', 'at', 'a', 'high', 'level', ',', 'its', 'challenges', ',', 'why', 'it', 'has']


In [3]:
all_tokens = sorted(set(preprocessed))
vocab_size = len(all_tokens)

print(f"Vocab length of dataset: {vocab_size}")

Vocab length of dataset: 6283


In [4]:
vocab = {token: integer for integer, token in enumerate(all_tokens)}

In [5]:
for idx, item in enumerate(vocab.items()):
    if idx > 10:
        break
    
    print(item)

('!', 0)
('$200', 1)
('$50', 2)
('$500', 3)
('$N', 4)
('&', 5)
('(', 6)
(')', 7)
(',', 8)
('.', 9)
('0', 10)


In [6]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {idx: item for item, idx in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [word for word in preprocessed if word.strip()]
        ids = [self.str_to_int[item] for item in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[id] for id in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [7]:
tokenizer = SimpleTokenizerV1(vocab)

text = """
DevOps teams have two primary roles in the ML model life cycle. First, they are the
people conducting and building operational systems as well as tests to ensure security,
performance, and availability of ML models. Second, they are responsible for CI/CD
pipeline management. Both of these roles require tight collaboration with data scien‐
tists, data engineers, and data architects. Tight collaboration is, of course, easier said
than done, but that is where MLOps can add value.
"""

encoded = tokenizer.encode(text)
print(f"Encoded Result: {encoded}", end="\n\n")

decoded = tokenizer.decode(encoded)
print(f"Decoded Result: {decoded}")

Encoded Result: [330, 5607, 3107, 5864, 4517, 5005, 3295, 5661, 601, 3915, 3636, 2155, 9, 432, 8, 5680, 1365, 5661, 4309, 1958, 1305, 1628, 4149, 5571, 1384, 6105, 1384, 5653, 5744, 2594, 5134, 8, 4318, 8, 1305, 1458, 4105, 601, 3922, 9, 879, 8, 5680, 1365, 4938, 2896, 217, 4367, 3745, 9, 199, 4105, 5678, 5005, 4915, 5719, 1852, 6155, 2171, 5096, 5738, 8, 2171, 2581, 8, 1305, 2171, 1359, 9, 984, 1852, 3488, 8, 4105, 2094, 8, 2494, 5046, 5657, 2437, 8, 1648, 5659, 3488, 6125, 604, 1674, 1171, 6009, 9]

Decoded Result: DevOps teams have two primary roles in the ML model life cycle. First, they are the people conducting and building operational systems as well as tests to ensure security, performance, and availability of ML models. Second, they are responsible for CI/CD pipeline management. Both of these roles require tight collaboration with data scien‐ tists, data engineers, and data architects. Tight collaboration is, of course, easier said than done, but that is where MLOps can add va

In [8]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab_size = len(all_tokens)

print(f"Vocab length of dataset: {vocab_size}")

Vocab length of dataset: 6285


In [9]:
vocab = {token: integer for integer, token in enumerate(all_tokens)}

In [10]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {idx: item for item, idx in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [word for word in preprocessed if word.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        ids = [self.str_to_int[item] for item in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[id] for id in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text


        

In [11]:
tokenizer = SimpleTokenizerV2(vocab)

text_1 = """
Machine learning operations (MLOps) is quickly becoming a critical component of
successful data science project deployment in the enterprise (Figure 1-1). It’s a process
that helps organizations and business leaders generate long-term value and reduce
risk associated with data science, machine learning, and AI initiatives. Yet it’s a rela‐
tively new concept; so why has it seemingly skyrocketed into the data science lexicon
overnight? This introductory chapter delves into what MLOps is at a high level, its
challenges, why it has become essential to a successful data science strategy in the
enterprise, and, critically, why it is coming to the forefront now.
"""

text_2 = """
At its core, MLOps is the standardization and streamlining of machine learning life
cycle management (Figure 1-2). But taking a step back, why does the machine learn‐
ing life cycle need to be streamlined? On the surface, just looking at the steps to go
from business problem to a machine learning model at a very high level, it seems
straightforward.
"""

text = " <|endoftext|> ".join((text_1, text_2))
print(text, end="\n\n")

encoded = tokenizer.encode(text)
print(f"Encoded Result: {encoded}", end="\n\n")

decoded = tokenizer.decode(encoded)
print(f"Decoded Result: {decoded}")


Machine learning operations (MLOps) is quickly becoming a critical component of
successful data science project deployment in the enterprise (Figure 1-1). It’s a process
that helps organizations and business leaders generate long-term value and reduce
risk associated with data science, machine learning, and AI initiatives. Yet it’s a rela‐
tively new concept; so why has it seemingly skyrocketed into the data science lexicon
overnight? This introductory chapter delves into what MLOps is at a high level, its
challenges, why it has become essential to a successful data science strategy in the
enterprise, and, critically, why it is coming to the forefront now.
 <|endoftext|> 
At its core, MLOps is the standardization and streamlining of machine learning life
cycle management (Figure 1-2). But taking a step back, why does the machine learn‐
ing life cycle need to be streamlined? On the surface, just looking at the steps to go
from business problem to a machine learning model at a very high